In [1]:
## Alex Feucht ASTR 427 Prof. Quinn
## HW 1, Due: 10/13
## AI Acknowledgement; Per the course syllabus, this is a disclosure of approved AI tool usage
## Claude 5.5 - debugging/documentation queries, Gemini - documentation queries, no generative uses

In [2]:
## Problem 1)
# Floating point representation
# For the float data type, write a program to empirically (i.e., by
# performing tests on the results of addition and subtraction operations
# within your program) determine the following “Machine constants” for
# your computer:
# (a) The smallest ϵ such that 1.0 − ϵ̸ = 1.0
# (b) The smallest ϵ such that 1.0 + ϵ̸ = 1.0
# (c) The maximum representable number
# (d) The minimum representable positive number
# Provide all the above answers to better than 10% accuracy, and
# comment on why the numbers you get are expected based on the
# IEEE 754 representation.

In [3]:
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
## 1.a)

index = 0 # Keeps track of n # epsilon iterations
val = np.float32(1.0) # Note: Power of 2 needed for clean bitwise traversal
epsilon = np.float32(1.0) # Initial guess to help the process, any 2^n works!

while val + epsilon / np.float32(2.0) != val: # Loop 1.0 + (1.0/2.0^n) until mantissa zeroes out
    epsilon = epsilon / np.float32(2.0) # each iteration updates epsilon smaller and smaller
    index += 1
    
print(f"Values match to precision limit, after {index} iterations.\nDecimal value of epsilon is {epsilon}\nFinal bitwise epsilon value of {epsilon.view(np.uint32):032b}")

Values match to precision limit, after 23 iterations.
Decimal value of epsilon is 1.1920928955078125e-07
Final bitwise epsilon value of 00110100000000000000000000000000


In [5]:
## 1.b)

# Similar concept to 1.a, but in the opposite direction!
index = 0
val = np.float32(1.0)
epsilon = np.float32(1.0)

while val - epsilon / np.float32(2.0) != val: # (+) -> (-)
    epsilon = epsilon / np.float32(2.0)
    index += 1
    
print(f"Values match to precision limit, after {index} iterations.\nDecimal value of epsilon is {epsilon}\nFinal bitwise epsilon value of {epsilon.view(np.uint32):032b}")

# Note: Difference of iterations/values can be chopped up to the exponentials not being centered on 0.0 but 1.0 when E = 0
# The value of 1.a) epsilon is 2 times larger than 1.b) epsilon and differs by the exponential bits only
# This is due to closing in on 1.0 from below instead of above, since 2^0=1, whereas the decimal numbers have an 
# inflection point at 0.0 between positives and negatives, exponents valued at 0 have an inflection point at 1.0, 
# then go into fractional positive values approaching 0.0, but floats go into the negatives, the two are asymmetric
# float values: (- , 0.0 , +), same exponent input: (- , 0.0, +) => evals to float value range: (0.0, 1.0, +) at limits/when E = 0

Values match to precision limit, after 24 iterations.
Decimal value of epsilon is 5.960464477539063e-08
Final bitwise epsilon value of 00110011100000000000000000000000


In [6]:
## 1.c)

index = 0
val = np.float32(1.0)

with np.errstate(over='raise'): # Allowing for overflows to pass via np package instead of error/warnings
    try:
        while True: # Loops while the error does not throw, handles exceptions/no runaway processes
            val = val *  np.float32(2.0) # Loop over powers of 2 ascending binary
            index += 1
    except FloatingPointError: # Will skip last FP error, passing the value before overflow
        pass

# Look up incrementer function, instead of finfo to do lookup

epsilon = np.finfo(np.float32).eps # Sort of a hack, numpy allows machine limit values to be referenced, this is smallest epsilon change
# that circumvents an overflow error. Technically, I could pull from earlier code in section 1.a
max_f32 = val * (np.float32(2.0) - epsilon) # My code overflows at 2^127 (the max), only does recordkeeping for the value before that
# The true max value would be the last value before overflow, plus one more power of 2 minus a corrective term to not let it go infinite/NaN

print(f"Iterations: {index}\nDecimal Value: {max_f32}\nBitwise Value: {max_f32.view(np.uint32):032b}")

# 1.0 chosen because non-zero, bit-wise this represents a 0'd out mantissa and leading digit 0 sign, exponent of 0, also a power of 2
# IEEE 754 definition of value 1: 0 (sign) 01111111 (exponent power) 00000000000000000000000 (mantissa)
# IEEE 754 definition of max float32: 0 (sign) 11111110 (exponent power) 11111111111111111111111 (mantissa)
# or 3.4028234663852886 x 10^38 in decimal (referenced internet/NR3 for these values for validation)
# Note: Every binary digit has flipped completely except for the sign, signifying a maximum finite value before infinity
# Sign of 1 or 0 and all 1's after is +-infinity, but finite values cap at 254 exponent, not 255, so the last exponent bit is 0
# Stored exponents go from 1-254, but are shifted to represent E's of -126 to +127 exponents for unsigned float32's

Iterations: 127
Decimal Value: 3.4028234663852886e+38
Bitwise Value: 01111111011111111111111111111111


In [7]:
#1.d)

index = 0
val = np.float32(1.0)

with np.errstate(under='raise'): # Underflow error protection instead of overflow
    try:
        while True:
            val = val / np.float32(2.0)
            index += 1
    except FloatingPointError:
        pass

epsilon = np.finfo(np.float32).eps
max_f32 = val / (np.float32(2.0) - epsilon)
print(f"Iterations: {index}\nDecimal Value: {max_f32}\nBitwise Value: {max_f32.view(np.uint32):032b}")

# Virtually identical to above code, just changed the * for / inside the while loop and max_32 conversion with epsilon term
# Results in a binary representation that is the lowest possible non-infinite value, if the last bit flipped would be inf
# Max float32 can only increase by +1 exponent before it goes +inf, min can only lose the last mantissa bit before -inf

Iterations: 149
Decimal Value: 1.401298464324817e-45
Bitwise Value: 00000000000000000000000000000001


In [8]:
## Problem 2)
# Floating point representation and rounding
# Write a method to sum the elements of an array using using only
# numpy.float32 operations (32 bit IEEE 754 floating point). For this
# part of the problem, avoid using the numpy internal sum method (see
# below). Generate an array of 20 million random numbers of type
# float32 (32 bit IEEE 754 floating point) between 0 and 1 (use an
# available method such as numpy.random.rand() in Python), and use
# your method to calculate the the sum of the numbers in the array
# (a) in the order as produced by rand(),
# (b) after being sorted from smallest to largest
# (c) after being sorted from largest to smallest.
# Explain the differences in the results from your sum method based on
# the limitations of the IEEE 754 precision. Also calculate the above
# sums using the internal numpy.ndarray.sum() method. Speculate on
# how the internal method differs from your implementation.

In [9]:
## 2.a) Random order sum (not using np.sum)

from numpy.random import RandomState # Consistent randomness state for validation testing
rng = np.random.default_rng(27) # Lucky number for seed

rand_array = rng.random(size=20000000, dtype=np.float32) # Generates np.random array, 20mil length over default interval [0.0, 1,0) via np documentation
print(f"Sampled Values: {rand_array}\nMax Array Value: {rand_array.max()}\nMin Array Value: {rand_array.min()}")

index = 0
summation = 0.0
while index != rand_array.size:
    summation += rand_array[index]
    index += 1
    #break
print(f"Sum: {summation} \nIndices Traversed: {index} \nArray Size: {rand_array.size}\nBitwise Sum: {summation.view(np.uint32):032b}")

Sampled Values: [0.00256932 0.6977362  0.2133851  ... 0.73657745 0.6946423  0.9327443 ]
Max Array Value: 0.9999999403953552
Min Array Value: 0.0
Sum: 10000908.0 
Indices Traversed: 20000000 
Array Size: 20000000
Bitwise Sum: 01001011000110001001101000001100


In [10]:
## 2.b) Ascending order, sort from lowest to highest

from numpy.random import RandomState # Consistent randomness state for validation testing
rng = np.random.default_rng(27) # Lucky number for seed

arr = rng.random(size=20000000, dtype=np.float32)
rand_array = np.sort(arr) # np.sort to arrange in order
print(f"Sampled Values: {rand_array}\nMax Array Value: {rand_array.max()}\nMin Array Value: {rand_array.min()}")

index = 0
summation = 0.0
while index != rand_array.size:
    summation += rand_array[index]
    index += 1
    #break
print(f"Sum: {summation} \nIndices Traversed: {index} \nArray Size: {rand_array.size}\nBitwise Sum: {summation.view(np.uint32):032b}")

Sampled Values: [0.0000000e+00 0.0000000e+00 5.9604645e-08 ... 9.9999988e-01 9.9999994e-01
 9.9999994e-01]
Max Array Value: 0.9999999403953552
Min Array Value: 0.0
Sum: 10167746.0 
Indices Traversed: 20000000 
Array Size: 20000000
Bitwise Sum: 01001011000110110010010111000010


In [11]:
## 2.c) Descending order, sort from highest to lowest

from numpy.random import RandomState # Consistent randomness state for validation testing
rng = np.random.default_rng(27) # Lucky number for seed

arr = rng.random(size=20000000, dtype=np.float32)
rand_array = np.flip(np.sort(arr)) # flips np.sort, effectively doing the reverse order
print(f"Sampled Values: {rand_array}\nMax Array Value: {rand_array.max()}\nMin Array Value: {rand_array.min()}")

index = 0
summation = 0.0
while index != rand_array.size:
    summation += rand_array[index]
    index += 1
    #break
print(f"Sum: {summation} \nIndices Traversed: {index} \nArray Size: {rand_array.size}\nBitwise Sum: {summation.view(np.uint32):032b}")

Sampled Values: [9.9999994e-01 9.9999994e-01 9.9999988e-01 ... 5.9604645e-08 0.0000000e+00
 0.0000000e+00]
Max Array Value: 0.9999999403953552
Min Array Value: 0.0
Sum: 8388608.0 
Indices Traversed: 20000000 
Array Size: 20000000
Bitwise Sum: 01001011000000000000000000000000


In [12]:
## 2.d?) Np array sum method + explanation of difference in methods

from numpy.random import RandomState # Consistent randomness state for validation testing
rng = np.random.default_rng(27) # Lucky number for seed

rand_array = rng.random(size=20000000, dtype=np.float32)
print(f"Sampled Values: {rand_array}\nMax Array Value: {rand_array.max()}\nMin Array Value: {rand_array.min()}")

summation = np.sum(rand_array) # using reference method

print(f"Sum: {summation} \nArray Size: {rand_array.size}\nBitwise Sum: {summation.view(np.uint32):032b}")

# The np.sum and all the other methods are not in total agreement. The first naive method of sorting randomly is close, ~400.0 off
# It seems like the randomly generated distribution of numbers is the most consistent after summation rounding errors
# Ascending order is the next closest, and descending is the worst performing one, at over ~16% off reference value.
# Ideally, with a perfect random distribution between 0.0 and 1.0, one would expect a value of 0.5, via statistical methods
# In reality, the generation of the numbers is pseudo-random, not perfectly random, a running summation cause mantissa bits to run off,
# and order matters when you are doing these kinds of operations. If the data starts truncating, precision is lost over time
# starting from the large numbers working down, the truncation is disproportionately larger than it would be starting from the small numbers.
# Only at the end does ascending start to reach the larger values, whereas descending starts with the largest numbers immediately

Sampled Values: [0.00256932 0.6977362  0.2133851  ... 0.73657745 0.6946423  0.9327443 ]
Max Array Value: 0.9999999403953552
Min Array Value: 0.0
Sum: 10000508.0 
Array Size: 20000000
Bitwise Sum: 01001011000110001001100001111100


In [13]:
## Problem 3)
# Roundoff error
# Numerically evaluate the expression (1−cos(x))/x2 in double precision
# for values of x around 10−7 and smaller. Explain the difference between
# the numerical results and the analytic limit as x → 0.
# Explain why trouble is occurring at x ∼ 10−7 when double precision is
# good to ∼ 10−15?

In [ ]:
## 3) 
# Analytic Evaluation:
# L'Hopital's eval, lim x->0 (1-cos(x))/x^2 produces 0/0 form
# L'Hopital twice gives value 1/2, global peak value at x=0, 
# discontinuity forms a hole graphically at this point (0,1/2)
# First derivative = (xsin(x)+2cos(x)-2)/x^3
# Second derivative = [(x^2-6)cos(x)-4xsin(x)+6]/x^4

def funct(x):
    return np.float64((1.0 - math.cos(x)) / (x ** 2.0))

exp = np.linspace(-7.0, -8.0, 10 ** 7) # variable iteration steps through exponent values, linear spacing to match float64
result = np.empty(exp.size, dtype=np.float64)

for i, e in enumerate(exp): # For the index i, exp -> e inside function calls from above
    result[i] = funct(10.0 ** e) # loops over the entire empty result array for exponents, sets values per index

np.set_printoptions(precision=17) # setting precision higher for readouts

analytic = np.float64(0.5) # 1/2 analytic sol.
index = np.argmin((np.abs(result - analytic))) # abs so results above or below 0.5 treated by magnitude
exact = result == analytic # makes sister array of any results that match exactly to analytic float64

print(np.column_stack((exp, result)))
print(f"Closest Value to Analytic Sol\n[Exponent, Function Value]: \n{np.column_stack((exp[index], result[index]))}")
print(f"# of 0.5 Valued Exp.: {exact.sum()}\nList of Exp. = Analytic: {np.column_stack((exp[exact], result[exact]))}")

# Plenty of near misses of the function vs. the analytic solution did occur with this code
# The results == analytic goes all the way out to double precision here, and at the bottom it shows 0 exact matches of 0.5
# Essentially, the computer is doing interpolation approximations and keeps jumping above or below 0.5, slowly approaching it
# The cos(x) and reciprocal x^2 terms definitely throws off precision for exceedingly smaller x values
# this is due to the x*x term having its own set of rounding errors associated with multiplying it out too
# It might get arbitrarily close up to machine precision, float64 here, but near the discontinuity at x = 0 gets out of control
# The function starts producing values of 0. after the closest approximation it can generate, and is also limited by the amount
# of linear spacings I take between -7.0 and -8.0 exponents. This could be fine tuned a bit to maybe get closer or cover smaller range

In [ ]:
## Problem 4)
# Interpolation
# (a) Write a program to read in a two column table from a file and per-
# form linear interpolation at an arbitrary point. You may assume
# that the data is evenly spaced in the independent variable (this
# makes it easier to determine which points to use for interpolation).
# (b) Use the program on the following input data: (available on the
# canvas site as hw_data.csv)
# x y
# 1 100
# 2 25
# 3 11.111111
# 4 6.25
# 5 4
# and provide an estimate of y at x = 4.75.
# (c) Either using your own routine or a package (e.g. scipy.interpolate)
# calculate interpolated values of y at 100 equally spaced points be-
# tween x = 1 and x = 5 using the above data with linear, quadratic,
# and 4th order polynomials, and plot the interpolated y vs. x.
# (Note: we are NOT performing a “fit” here; we want polynomi-
# als that go exactly through all the given data points.) Give an
# estimate of y at x = 4.75 for each of these polynomial orders.
# (d) The actual function tabulated above is y = 100
# x2 . Compare the
# actual value at x = 4.75 with the linear interpolation and the 4th
# order polynomial interpolation, and comment on why one is more
# accurate than the other.

In [ ]:
## 4.a/b)
# NR3 Ref Page 180 PDF, Chap 3/4, 4th order N or Taylor expansion, Splines
from scipy.interpolate import interp1d, lagrange

df = pd.read_csv('hw1_data.csv', header=0, names=["x", "y"])
x = df.iloc[:, 0].to_numpy(dtype=np.float64)
y = df.iloc[:, 1].to_numpy(dtype=np.float64)

f = interp1d(x, y, kind="linear")
xq = np.linspace(x.min(), x.max(), 5)
yq = f(xq)

print(f"Value of 4.75 interpolation Estimate: {f(4.75)}")

fig, ax = plt.subplots(figsize=(8,6))
ax.plot(xq, yq, "-", label="1D Interpolation")
ax.plot(x, y, "o", c='red', label="Data Points")
ax.plot(4.75, f(4.75), "*", markersize=14 ,label="x = 4.75")
ax.set_xlabel(df.columns[0])
ax.set_ylabel(df.columns[1])
ax.grid(True, alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
## 4.c)

xq2 = np.linspace(1.0, 5.0, 100)

f_linear = interp1d(x, y, kind = "linear")
f_quadratic = interp1d(x, y, kind = "quadratic")
f_4th = interp1d(x, y, kind=4)
f_lagrange = lagrange(x, y)

fig, ax2 = plt.subplots(figsize=(14,8))

ax2.plot(x, y, "o", zorder=2, c='red', label="Data Points")
ax2.plot(4.75, (100 / (4.75)**2), "*", c='green', zorder=2, markersize=14 ,label=f"x=4.75, y={(100 / (4.75)**2):.4f}")
ax2.plot(xq2, 100 / xq2**2, "-", c='magenta', zorder=1, label="Actual 100/x^2")
ax2.plot(xq2, f_linear(xq2), zorder=1, c='blue', label="Linear")
ax2.plot(xq2, f_quadratic(xq2), c='orange', zorder=1, label="Quadratic")
ax2.plot(xq2, f_4th(xq2), zorder=2, c='green', alpha=0.6, linewidth=3, label="Fourth Order") # alpha + width due to overlap
ax2.plot(xq2, f_lagrange(xq2), c='black', zorder=1, label="Lagrange")

ax2.set_xlabel(df.columns[0])
ax2.set_ylabel(df.columns[1])
ax2.grid(True, alpha=0.2)
ax2.legend()
fig.tight_layout()
plt.show()

In [ ]:
## 4.d)

# Only asks for Linear and 4th Order comparrison, made all available from above method, and Lagrange reference
print(f"Actual Value 100/x^2, x = 4.75:\n{100/(4.75)**2}")
print(f"Linear Iterpolation/Difference from actual:\n{f_linear(4.75)}, {abs((100/(4.75)**2) - f_linear(4.75))}")
# print(f"Quadratic Interpolation/Difference from actual:\n{f_quadratic(4.75)}, {abs((100/(4.75)**2) - f_quadratic(4.75))}")
print(f"4th Order Polynomial/Difference from actual:\n{f_4th(4.75)}, {abs((100/(4.75)**2) - f_4th(4.75))}")
# print(f"Lagrange Interpolation/Difference from actual:\n{f_lagrange(4.75)}, {abs((100/(4.75)**2) - f_lagrange(4.75))}")

# The Linear Interpolation is much better suited for this application than the 4th Order polynomial one, likely due
# to the piecewise nature of this plotting of regularly spaced points. Linear can draw a line between points 
# without as much having to go on either side of the real function values to try and fit the polynomial curve's "stiffness"
# to the plotted points. If it were not so discrete of a function, or more continuous without big gaps, maybe it's a different story